# Step 3 of RAG: Retrieving Relevant Context Based on User Query

**RAG** (Retrieval-Augmented Generation) pipeline recap:

1. Document Loading & Chunking (Step 1)
2. Storing chunks in a Vector Database (Step 2)
3. **Retrieval — finding the most relevant chunks for a user's question** ← *this notebook*
4. Passing retrieved chunks to an LLM to generate the final answer

This notebook is self-contained — it quickly recreates and stores the chunks from Steps 1-2,
then focuses entirely on **retrieval**: how to turn a user's question into the right pieces of
context, and how to filter out irrelevant results.

### What you will learn
1. Recap: chunk and store documents in ChromaDB
2. Basic retrieval with `query()` and `n_results`
3. Understanding distance vs. similarity scores
4. Filtering out weak/irrelevant matches with a similarity threshold
5. Combining semantic search with metadata filters (`where`)
6. Building a reusable `retrieve_context()` function
7. Formatting retrieved chunks into an LLM-ready prompt (preview of Step 4)
8. Testing retrieval with paraphrased, real-world-style questions

### Setup — Install Required Libraries

In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

## 1. Recap — Chunk and Store Documents (Steps 1 & 2)

Two short documents, split into chunks, stored in ChromaDB with metadata. This mirrors the
previous notebooks in this series, condensed into one cell so this notebook runs standalone.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""])

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name="rag_chunks", embedding_function=hf_embedder)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total chunks stored: 13


## 2. Basic Retrieval — `query()` and `n_results`

Given a user's question, ChromaDB embeds it and returns the `n_results` closest chunks.

In [3]:
user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

Distance: 0.3244 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
------------------------------------------------------------
Distance: 0.6356 | Source: ai_overview.txt | Chunk #7
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
------------------------------------------------------------
Distance: 0.6794 | Source: embeddings_in_practice.txt | Chunk #2
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------


**Discussion Point:** `n_results` controls how many chunks come back. Too few might miss useful
context; too many adds noise (and costs more tokens later when passed to an LLM). `n_results=3` to `5`
is a common starting point.

## 3. Understanding Distance vs. Similarity

ChromaDB's default distance metric is **cosine distance**, where:

$$
\text{cosine\_distance} = 1 - \text{cosine\_similarity}
$$

So a **smaller distance** means **higher similarity** (more relevant). We can convert back to a
similarity score to make results easier to reason about.

In [4]:
def distance_to_similarity(distance):
    return 1 - distance

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    similarity = distance_to_similarity(distance)
    print(f"Similarity: {similarity:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")

Similarity: 0.6756 | Source: ai_overview.txt | Chunk #6
Similarity: 0.3644 | Source: ai_overview.txt | Chunk #7
Similarity: 0.3206 | Source: embeddings_in_practice.txt | Chunk #2


## 4. Filtering Out Weak Matches with a Similarity Threshold

Not every chunk returned is actually relevant — if a user asks something completely unrelated to our
documents, ChromaDB will still return its "closest" chunks, even if none of them are a good match.
A **similarity threshold** lets us discard weak matches instead of blindly trusting `n_results`.

In [5]:
def retrieve_with_threshold(query, collection, n_results=5, similarity_threshold=0.3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    filtered = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            filtered.append({"text": doc, "metadata": meta, "similarity": similarity})

    return filtered

# A relevant query
relevant_matches = retrieve_with_threshold("What is deep learning?", collection, similarity_threshold=0.3)
print(f"Relevant query -> {len(relevant_matches)} chunk(s) passed the threshold\n")
for m in relevant_matches:
    print(f"Similarity: {m['similarity']:.4f} | {m['text'][:80]}...")

Relevant query -> 4 chunk(s) passed the threshold

Similarity: 0.6346 | Deep Learning takes Machine Learning further by using neural networks with many ...
Similarity: 0.4946 | Machine Learning is a subset of AI where systems learn patterns from data instea...
Similarity: 0.3875 | Artificial Intelligence, or AI, is the field of computer science focused on buil...
Similarity: 0.3682 | understanding language, recognizing images, making decisions, and learning from ...


In [6]:
# An unrelated query - notice fewer (or zero) chunks pass the threshold
unrelated_matches = retrieve_with_threshold("What is the best recipe for chocolate cake?", collection, similarity_threshold=0.3)
print(f"Unrelated query -> {len(unrelated_matches)} chunk(s) passed the threshold")

if len(unrelated_matches) == 0:
    print("No relevant context found - a real RAG system should say 'I don't know' here instead of guessing.")

Unrelated query -> 0 chunk(s) passed the threshold
No relevant context found - a real RAG system should say 'I don't know' here instead of guessing.


**Discussion Point:** This threshold check is what prevents a RAG system from confidently answering
questions its documents actually know nothing about — a key technique for reducing hallucination.

## 5. Combining Semantic Search with Metadata Filters (`where`)

Sometimes we want to restrict retrieval to a specific source, category, or date range, in addition
to searching by meaning.

In [7]:
results = collection.query(
    query_texts=["how does search work with meaning instead of keywords"],
    n_results=2,
    where={"source": "embeddings_in_practice.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Similarity: {distance_to_similarity(distance):.4f} | Source: {meta['source']}")
    print(doc)
    print("-" * 60)

Similarity: 0.6371 | Source: embeddings_in_practice.txt
allowing the system to find results that match the meaning of the query, not just its exact words.
------------------------------------------------------------
Similarity: 0.6114 | Source: embeddings_in_practice.txt
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
------------------------------------------------------------


## 6. Building a Reusable `retrieve_context()` Function

Let's combine everything above (top-k, threshold, optional filter) into one clean function we can
reuse for any query.

In [8]:
def retrieve_context(query, collection, n_results=5, similarity_threshold=0.25, source_filter=None):
    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }
    if source_filter:
        query_kwargs["where"] = {"source": source_filter}

    results = collection.query(**query_kwargs)

    context_chunks = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })

    return context_chunks

context = retrieve_context("What is machine learning?", collection)
for c in context:
    print(c)

{'text': 'Machine Learning is a subset of AI where systems learn patterns from data instead of following\nhardcoded rules. Instead of programming every decision by hand, we feed the system many examples,\nand it learns the underlying pattern on its own.', 'source': 'ai_overview.txt', 'chunk_index': 3, 'similarity': 0.7744}
{'text': 'Deep Learning takes Machine Learning further by using neural networks with many layers. These\nlayered networks can automatically discover complex patterns in large amounts of data, such as', 'source': 'ai_overview.txt', 'chunk_index': 4, 'similarity': 0.5253}
{'text': 'Artificial Intelligence, or AI, is the field of computer science focused on building systems\nthat can perform tasks which normally require human intelligence. These tasks include', 'source': 'ai_overview.txt', 'chunk_index': 1, 'similarity': 0.4885}
{'text': 'Artificial Intelligence: An Overview', 'source': 'ai_overview.txt', 'chunk_index': 0, 'similarity': 0.3746}
{'text': 'understanding l

## 7. Formatting Retrieved Chunks into an LLM-Ready Prompt (Preview of Step 4)

The final job of retrieval is to hand off clean, well-labeled context to a language model. Here's
how that context typically gets formatted into a prompt.

In [9]:
def format_context_for_llm(query, context_chunks):
    if not context_chunks:
        return f"Question: {query}\n\nNo relevant context was found in the knowledge base."

    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )

    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
{context_text}

Question: {query}

Answer:"""
    return prompt

context = retrieve_context("How does RAG combine search with a language model?", collection)
final_prompt = format_context_for_llm("How does RAG combine search with a language model?", context)

print(final_prompt)

Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
[Source: ai_overview.txt, chunk 6]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG

[Source: embeddings_in_practice.txt, chunk 2]
allowing the system to find results that match the meaning of the query, not just its exact words.

[Source: embeddings_in_practice.txt, chunk 1]
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,

[Source: embeddings_in_practice.txt, chunk 0]
Embeddings and Vector Search in Practice

[Source: ai_overview.txt, chunk 7]
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, gro

**Discussion Point:** This exact `final_prompt` string is what Step 4 would send to a language model
(free or paid) to generate the final, grounded answer — completing the RAG pipeline.

## 8. Testing Retrieval with Paraphrased, Real-World-Style Questions

A good retrieval system should still work even when the user's wording doesn't match the document's
wording exactly.

In [10]:
test_queries = [
    "Why do companies use vector databases?",
    "What's the difference between AI and machine learning?",
    "How do chatbots understand differently worded questions?",
]

for q in test_queries:
    print(f"Query: {q}")
    matches = retrieve_context(q, collection, n_results=2)
    for m in matches:
        print(f"   [{m['similarity']}] ({m['source']}) {m['text'][:90]}...")
    print()

Query: Why do companies use vector databases?
   [0.4543] (embeddings_in_practice.txt) Embeddings and Vector Search in Practice...
   [0.4013] (embeddings_in_practice.txt) Vector databases such as ChromaDB store these embeddings and allow fast similarity search ...

Query: What's the difference between AI and machine learning?
   [0.6668] (ai_overview.txt) Artificial Intelligence, or AI, is the field of computer science focused on building syste...
   [0.6572] (ai_overview.txt) Machine Learning is a subset of AI where systems learn patterns from data instead of follo...

Query: How do chatbots understand differently worded questions?
   [0.4263] (embeddings_in_practice.txt) allowing the system to find results that match the meaning of the query, not just its exac...
   [0.3312] (embeddings_in_practice.txt) This is especially powerful for customer support systems, where a user might phrase a ques...



## Key Takeaways (Recap for Students)

| Concept | What It Does |
|---|---|
| `n_results` | How many chunks to retrieve per query |
| Distance vs. similarity | `similarity = 1 - distance` for cosine distance in ChromaDB |
| Similarity threshold | Filters out weak/irrelevant matches instead of trusting every result |
| `where` filter | Restricts retrieval to a specific source/category alongside semantic search |
| `retrieve_context()` | Reusable function combining top-k, threshold, and optional filtering |
| Prompt formatting | Turns retrieved chunks into a clean, labeled context block for an LLM |

Retrieval is the heart of RAG — get this step right (good chunking + good thresholds) and the final
answer from Step 4 will be far more accurate and far less likely to hallucinate.

### Practice Exercises
1. Lower `similarity_threshold` to `0.1` and re-run the "chocolate cake" query — how many chunks pass now?
   What does that tell you about picking a good threshold?
2. Add a `max_chunks` parameter to `retrieve_context()` that limits the *total combined character length*
   of returned chunks (useful for staying under an LLM's context window).
3. Add a third document to the collection and confirm `retrieve_context()` can pull chunks from it too.

In [11]:
import chromadb

# Create ChromaDB client
client = chromadb.Client()

# Create collection
collection = client.get_or_create_collection("exercise1")

# Add documents
documents = [
    "Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter.",
    "Vanilla cake is made with flour, sugar, eggs, butter, and vanilla extract.",
    "Pizza is made with dough, tomato sauce, cheese, and toppings.",
    "Chocolate cookies are made with flour, sugar, cocoa powder, and chocolate chips.",
    "Pasta is usually cooked in boiling water and served with sauce."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

collection.add(
    documents=documents,
    ids=ids
)


def retrieve_context(
    collection,
    query,
    n_results=5,
    similarity_threshold=0.1
):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    documents = results["documents"][0]
    distances = results["distances"][0]

    filtered = []

    for doc, distance in zip(documents, distances):

        similarity = 1 - distance

        print(f"Similarity: {similarity:.3f}")
        print(f"Document: {doc}\n")

        if similarity >= similarity_threshold:
            filtered.append(doc)

    return filtered


# Query
context = retrieve_context(
    collection,
    "chocolate cake",
    n_results=5,
    similarity_threshold=0.1
)

print("=" * 50)
print("FINAL RESULTS")
print("=" * 50)

print(f"Number of chunks returned: {len(context)}")

for i, chunk in enumerate(context, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


Similarity: 0.358
Document: Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter.

Similarity: -0.046
Document: Chocolate cookies are made with flour, sugar, cocoa powder, and chocolate chips.

Similarity: -0.094
Document: Vanilla cake is made with flour, sugar, eggs, butter, and vanilla extract.

Similarity: -0.709
Document: Pizza is made with dough, tomato sauce, cheese, and toppings.

Similarity: -0.714
Document: Pasta is usually cooked in boiling water and served with sauce.

FINAL RESULTS
Number of chunks returned: 1

--- Chunk 1 ---
Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter.


In [12]:
import chromadb

# Create ChromaDB client
client = chromadb.Client()

# Create collection
collection = client.get_or_create_collection("exercise2")

# Add documents
documents = [
    "Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter. It is baked in an oven until fully cooked.",
    "Vanilla cake is made with flour, sugar, eggs, butter, and vanilla extract. It has a soft texture and sweet vanilla flavor.",
    "Chocolate cookies are made with flour, sugar, cocoa powder, butter, and chocolate chips. They are baked until slightly crispy.",
    "Pizza dough is made using flour, water, yeast, salt, and olive oil. The dough is allowed to rise before baking.",
    "Pasta is cooked in boiling water and can be served with tomato sauce, cheese, vegetables, or meat."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

collection.add(
    documents=documents,
    ids=ids
)


def retrieve_context(
    collection,
    query,
    n_results=5,
    similarity_threshold=0.1,
    max_chunks=200
):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    documents = results["documents"][0]
    distances = results["distances"][0]

    filtered = []
    total_characters = 0

    for doc, distance in zip(documents, distances):

        similarity = 1 - distance

        if similarity >= similarity_threshold:

            # Check whether adding this document
            # would exceed max_chunks
            if total_characters + len(doc) > max_chunks:
                break

            filtered.append(doc)
            total_characters += len(doc)

    return filtered


# Query
context = retrieve_context(
    collection,
    "chocolate cake",
    n_results=5,
    similarity_threshold=0.1,
    max_chunks=200
)


print("=" * 50)
print("RETRIEVED CONTEXT")
print("=" * 50)

total_characters = 0

for i, chunk in enumerate(context, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

    total_characters += len(chunk)


print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)

print(f"Number of chunks: {len(context)}")
print(f"Total characters: {total_characters}")
print("Maximum allowed: 200")


RETRIEVED CONTEXT

--- Chunk 1 ---
Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter. It is baked in an oven until fully cooked.

SUMMARY
Number of chunks: 1
Total characters: 116
Maximum allowed: 200


In [13]:
import chromadb

# Create ChromaDB client
client = chromadb.Client()

# Create collection
collection = client.get_or_create_collection("exercise3")


# -----------------------------
# DOCUMENT 1
# -----------------------------

document1 = """
Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter.
It is baked in an oven and usually served as a dessert.
"""


# -----------------------------
# DOCUMENT 2
# -----------------------------

document2 = """
Pasta is cooked in boiling water.
It can be served with tomato sauce, cheese, vegetables, or meat.
"""


# -----------------------------
# DOCUMENT 3
# -----------------------------

document3 = """
Pizza dough is made using flour, water, yeast, salt, and olive oil.
The dough is allowed to rise before it is baked.
Pizza can be topped with cheese, tomato sauce, vegetables, and meat.
"""


# Add all three documents
collection.add(
    documents=[
        document1,
        document2,
        document3
    ],
    ids=[
        "document_1",
        "document_2",
        "document_3"
    ],
    metadatas=[
        {"source": "cake.txt", "category": "dessert"},
        {"source": "pasta.txt", "category": "food"},
        {"source": "pizza.txt", "category": "food"}
    ]
)


def retrieve_context(
    collection,
    query,
    n_results=3,
    similarity_threshold=0.1,
    where=None
):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where
    )

    documents = results["documents"][0]
    distances = results["distances"][0]
    metadatas = results["metadatas"][0]

    filtered = []

    for doc, distance, metadata in zip(
        documents,
        distances,
        metadatas
    ):

        similarity = 1 - distance

        print(f"Similarity: {similarity:.3f}")
        print(f"Source: {metadata['source']}")

        if similarity >= similarity_threshold:
            filtered.append(doc)

    return filtered


# -----------------------------
# TEST 1: Chocolate cake
# -----------------------------

print("\n" + "=" * 60)
print("QUERY 1: chocolate cake")
print("=" * 60)

context = retrieve_context(
    collection,
    "chocolate cake",
    n_results=3,
    similarity_threshold=0.1
)

for i, chunk in enumerate(context, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


# -----------------------------
# TEST 2: Pizza
# -----------------------------

print("\n" + "=" * 60)
print("QUERY 2: pizza dough")
print("=" * 60)

context = retrieve_context(
    collection,
    "pizza dough",
    n_results=3,
    similarity_threshold=0.1
)

for i, chunk in enumerate(context, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


# -----------------------------
# TEST 3: Search only food
# -----------------------------

print("\n" + "=" * 60)
print("QUERY 3: food documents only")
print("=" * 60)

context = retrieve_context(
    collection,
    "food",
    n_results=3,
    similarity_threshold=0.1,
    where={"category": "food"}
)

for i, chunk in enumerate(context, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)



QUERY 1: chocolate cake
Similarity: 0.350
Source: cake.txt
Similarity: -0.802
Source: pizza.txt
Similarity: -0.822
Source: pasta.txt

--- Chunk 1 ---

Chocolate cake is made with cocoa powder, flour, sugar, eggs, and butter.
It is baked in an oven and usually served as a dessert.


QUERY 2: pizza dough
Similarity: 0.446
Source: pizza.txt
Similarity: -0.505
Source: pasta.txt
Similarity: -0.720
Source: cake.txt

--- Chunk 1 ---

Pizza dough is made using flour, water, yeast, salt, and olive oil.
The dough is allowed to rise before it is baked.
Pizza can be topped with cheese, tomato sauce, vegetables, and meat.


QUERY 3: food documents only
Similarity: -0.449
Source: pasta.txt
Similarity: -0.639
Source: pizza.txt
